In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, fixed


# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d, build_tissue
from render import render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap, NormalizeData
from optics import kryostat, psf_project, mask_collapse, detector
from parameter import (P, CellGeometry, MarkerPanel, PanelMarker, CellType, Detector,
                       dapi_marker,
                       BlobNoise, ClusterNoise, NetworkNoise, FibreNoise, SheetNoise,
                       Optics, TissueGeometry)
from artifacts import build_artifacts
import config as cfg
from viewer3d import scene3d


%matplotlib inline

In [ ]:
SEED       = cfg.SEED
N_CAND     = cfg.N_CAND
SIZE       = cfg.SIZE
TILE       = cfg.TILE
K          = cfg.K
L_MIN, L   = cfg.L_MIN, cfg.L
UM_PER_VOX = cfg.UM_PER_VOX
Z_RATIO    = cfg.Z_RATIO
SPACING    = cfg.SPACING
VOL        = cfg.CELL_VOL
TISSUE_VOL = cfg.TISSUE_VOL
N_POOLS    = cfg.N_POOLS


GEOM = CellGeometry()
DETECTOR = Detector()


PANEL = MarkerPanel(
    Markers = dict(
        DAPI = dapi_marker(),


        r = PanelMarker(name="r", fluorophore="APC", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.80, s=1.40, mu= .55, width=.45, sharp=5.0,
                             scale=.35, clust=2.00, fill=.22, soft=.25),
                FibreNoise  (w=.30, s=1.30, mu= .20, width=.70, sharp=4.0,
                             lam=.25, length=6.0),
                NetworkNoise(w=.40, s=1.30, mu= .50, width=.55, sharp=3.5,
                             scale=.80, coherence=.40),
            ], artifact_affinity=1.2),

        g = PanelMarker(name="g", fluorophore="FITC", amp=2.0, polarity=0.2,
            noise_components=[
                BlobNoise   (w=.50, s=1.50, mu= .10, width=.60, sharp=7.5,
                             scale=.45),
                SheetNoise  (w=.60, s=1.50, mu= .40, width=1.20, sharp=6.0,
                             lam=1.90, coherence=.35, length=6.0),
                NetworkNoise(w=.90, s=1.40, mu= .20, width=1.10, sharp=7.0,
                             scale=1.00, coherence=.60),
            ], artifact_affinity=0.2)
    )
)


N_POOLS=PANEL.n_pools()

# 4) DRAW THE TAPE
tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)

# 1) Synthetic cell generation

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, GEOM, size=VOL, spacing=SPACING, i=0, L=L, l_min=L_MIN)

In [ ]:
# `opt` did not exist yet at this point in the old project's notebook layout -- this section
# builds its own Optics the same way the tissue section does further down, so the sandbox is
# self-contained and can be re-run on its own.
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)

fig = scene3d(cell=cell["cell"], nuc=cell["nuc"], d=cell["d"], tau=cell["tau"],
              opt=optics, colour_by="flat", colour="#AB63FA",
              title="one cell -- drag to rotate")
print(fig._viewer_info)
fig.show()


In [ ]:
fig = plot_surface_xyz_inline(cell = cell, geom = GEOM)
fig = plot_surface_xyz_html(cell = cell, geom = GEOM,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:


IMG = (VOL[1], VOL[2], len(PANEL.names))
tape.drawSensor(shape=IMG)

# edge_softness=0 -> clipped EXACTLY at the plasma membrane. All the softness in the final
# image is made by psf_project and detector, not baked into the object.
# For this test, express at level 1 for every marker
SOLO = CellType(name="solo", Geometry=GEOM,
                Expression={k: P(0., 2., .05, 1.0, f"expr {k}") for k in PANEL.names})
img = render_image(tape, PANEL, cell, spacing=SPACING, geom=GEOM,
                   um_per_vox=UM_PER_VOX, edge_softness=0.0,
                   expression={k: SOLO.level(k) for k in PANEL.names})

rgb = to_rgb(cell["cell"], img)

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = GEOM.ELONG.v,
                        polar_deg = GEOM.POLAR_DEG.v, azim_deg = GEOM.AZIM_DEG.v, roll_deg = GEOM.ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)


subs = {marker: kryostat(v, optics) for marker, v in img.items()}   # keep BOTH vol and z

img_psf = np.stack([psf_project(v, z, optics, PANEL.Markers[name].fluorophore)
                    for name, (v, z) in subs.items()], -1)

In [ ]:
sub, sub_z = kryostat(cell["cell"], optics)
sub_n, sub_n_z = kryostat(cell["nuc"], optics)


plt.imshow(NormalizeData(img_psf))
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
# Pre-normalize
img_psf_norm = (img_psf - np.min(img_psf)) / (np.max(img_psf) - np.min(img_psf))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

### 1.2.2) 2d Projection + Detector Model

In [ ]:
markers = list(subs.keys())

img_adu = detector(img_psf, PANEL, DETECTOR, optics, tape, markers)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(NormalizeData(img_psf), origin="lower")
ax[0].set_title("clean projection (object x PSF)")
ax[1].imshow(NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0)), origin="lower")
ax[1].set_title("detector frame (AF + shot + read)")
ax[2].imshow(NormalizeData(img_adu[..., 0]), cmap="magma", origin="lower")
ax[2].set_title(f"channel 0 ({markers[0]}, {PANEL.Markers[markers[0]].fluorophore}) in ADU")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Pre-normalize
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

# 2) Synthetic tissue generation

## 2) Tissue directionality

In [ ]:
from parameter import TissueArchitecture, Vasculature, Ordered
from scene import build_tissue_V2
import copy, dataclasses

In [ ]:
tape.drawTissueDir(shape = TISSUE_VOL, n_cand = N_CAND, Pool = N_POOLS)
TG = TissueGeometry(COVER=0.90, MIN_DIST=10.0)

TISSUE_CELL = CellGeometry(RADIUS=8.0, ROUGH=0.15, ELONG=1.6, NUC_FRAC=0.40, RIM=0.0,
                           NUC_OFFSET=0.8, ALIGN=1.0)

def _expr(**levels):
    return {k: P(0., 2., .05, float(v), f"expr {k}",
                 comment="expression level; 0 = negative, 1 = the panel's nominal brightness")
            for k, v in levels.items()}

# PANEL has exactly two non-DAPI markers (r = APC, g = FITC), so with three cell types the
# clearest panel a 2-marker system can show is one positive for each marker alone plus one
# double positive -- every combination the panel can actually distinguish. OFF is not literally
# 0: a small floor (0.10) keeps each channel's noise model engaged instead of switching it off,
# which is what a real negative population looks like next to true background.
ON, OFF = 1.7, 0.10
CELLTYPES = {
    "stroma": CellType(name="stroma", Color="red", Geometry=copy.deepcopy(TISSUE_CELL),
                       Expression=_expr(DAPI=3.0, r=ON, g=OFF)),        # marker A only
    # NUC_ELONG is the nucleus's OWN aspect ratio, independent of the cell body's ELONG (see
    # parameter.CellGeometry.NUC_ELONG) -- a spindle-shaped myocyte does not carry a needle for
    # a nucleus. 1.0 = round is the structural null; the fibre still narrows it a little through
    # the containment clip against the RIM-0 membrane, which is realistic (real myonuclei are
    # flattened ellipsoids for exactly that reason) -- measured aspect ratio 13.7 -> 2.8 median.
    "myocyte": CellType(name="myocyte", Color="green",
                        Geometry=dataclasses.replace(copy.deepcopy(TISSUE_CELL), ELONG=6.0,
                                                     NUC_ELONG=1.0, ALIGN=1.0, NUC_CORR=0.0, RADIUS=12.0),
                        Expression=_expr(DAPI=3.0, r=OFF, g=ON)),       # marker B only
    "epithelium": CellType(name="epithelium", Color="blue",
                           Geometry=dataclasses.replace(copy.deepcopy(TISSUE_CELL), ELONG=1.2, ALIGN=1.0, NUC_CORR = 0.0),
                           Expression=_expr(DAPI=3.0, r=ON, g=ON)),     # A + B, double positive
}
FRACTIONS = [0.5, 0.25, 0.25]


ARCH = TissueArchitecture(
    # the prior field: an isotropic random flow, and the floor a structure's W has to beat.
    # This is what interstitial cells follow, and what FLOW="field" structures grow along.
    NOISE_W=1.0, SCALE_UM=7.0, SCALE_Z_UM=5.0, CURV=0.0,
    # the carve. MARGIN_UM is the guaranteed stromal collar; the noise is smeared ALONG the flow
    # so the tissue outline frays in the same direction the structures are elongated.
    MARGIN_UM=1.0, EDGE_NOISE=5.0, EDGE_SCALE_UM=4.0, LIC_STRETCH=3.0, MIN_ISLAND=0.002,
    TANGENCY=1.0, TANGENCY_Z_DEG=0.0, BAND=0.5, COMPACT=0.0,
    Structures=[
        # vessels. FLOW="axis" -- a duct runs straight, so its metric is uniform along its own
        # direction: the equilibrium shape is exactly an ellipsoid of ratio ASPECT, and it is the
        # CLOSED-FORM case, costing no graph at all. ASPECT is the axis ratio directly (this is
        # NOT the old ELONG, which was a stretch of ratio ELONG^1.5).
        Vasculature(name="vessel", W=7., FRAC=0.15, SIZE_UM=4.0, ASPECT=6.0, ROUGH=0.20,
                    BUMP_SCALE_UM=3.0, SPREAD=0.9, N_MAX=24, EDGE_UM=1.0, ALIGN=1.0,
                    FLOW="axis", DIR_DEG=-40.0, TILT_DEG=55.0, WALL_IN=0.45,
                    PHASE_DEG=90.0, PHASE_TILT_DEG=0.0, SIMILARITY=1.0, DENSITY=2.,
                    Profile={"epithelium": 4.0, "stroma": -0.5}),
        # the ordered muscle. FLOW="field" grows it along the prior flow, so it SNAKES with the
        # tissue grain instead of being a straight ellipsoid -- but SIMILARITY is what decides
        # HOW MUCH: it slerps that flow toward (DIR_DEG, TILT_DEG) before growing. At 0 the
        # bundle follows the raw isotropic noise and has essentially no direction (measured
        # nematic order 0.22); at 1 it is the declared axis exactly (0.87). 0.8 gives 0.71 --
        # it meanders, but it clearly runs somewhere, which is what muscle looks like.
        # Costs a geodesic solve, done downsampled (~0.9 s).
        Ordered(name="muscle", W=10., FRAC=0.65, SIZE_UM =50.0, ASPECT=3.0, ROUGH=0.10,
                BUMP_SCALE_UM=4.0, SPREAD=0.95, N_MAX=24, EDGE_UM=1.5, ALIGN=1.0,
                FLOW="field", SIMILARITY=1.0, DIR_DEG=25.0, TILT_DEG=70.0, DENSITY=2.,
                Profile={"myocyte": 4.0, "stroma": -1.0}),
    ])

In [ ]:
t_img, labels, nuc_labels, tau_img, types, info = build_tissue_V2(
    tape=tape, TG=TG, shape=TISSUE_VOL, base_geom=TISSUE_CELL, spacing=SPACING,
    Panel=PANEL, CellTypes=CELLTYPES, Fractions=FRACTIONS, um_per_vox=UM_PER_VOX,
    L=L, l_min=L_MIN, ARCH=ARCH)

In [ ]:
cmap = {name: ct.Color for name, ct in CELLTYPES.items()}
# PLot tissue mask
mask = info['support'][0].astype(int)
print(info['support'].shape[0])
cv = np.array([info["centres_vox"][n] for n in info["labels_present"]])
for i in range(info['support'].shape[0]-1):
    mask += info['support'][i].astype(int)
    plt.scatter(cv[i,0], cv[i,1], c = cmap[types[i+1]])
plt.imshow(mask)

### Interactive 3D view of the tissue


In [ ]:
cmap = {name: ct.Color for name, ct in CELLTYPES.items()}
col_by_id = {n: cmap[types[n]] for n in info["labels_present"]}
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)

# Same viewer as the single-cell sandbox in 1.1 (`viewer3d.scene3d`), now over the whole labelled
# block: one mesh for every cell membrane and one for every nucleus (`labels`/`nuc_labels` share
# the same ids, see build_tissue_V2), coloured by CELL TYPE via `colours` -- a dict keyed by label
# id rather than the default per-id rainbow -- so it reads the same way the "Cells -- typed and
# oriented" panel further down does. `edges=True` draws each cell's own triangulation on top as a
# thin dark wireframe -- with same-type neighbours sharing a fill colour, that outline is what
# actually tells two touching cells of the same type apart.
fig = scene3d(labels=labels, nuc_labels=nuc_labels, opt=optics,
              colour_by="instance", colours=col_by_id, opacity=0.45,
              edges=True, edge_colour="#111820", edge_width=1.0, edge_opacity=0.55,
              section_um=optics.section_um,
              title="tissue block -- coloured by cell type, drag to rotate")
print(fig._viewer_info)
fig.show()


In [ ]:
from plot import (field_lic, psi_rgb, comp_rgb, tilt_rgb, zones, zone_edges,
                  outline, palette, zone_palette, COH_CMAP)
from tissue_direction import field_plane

# The field is already in `info`: build_tissue computed it and used it to orient and type these
# very cells. Nothing is re-derived here, so the plots show what actually drove the image.
if info["field"] is None:
    raise RuntimeError("no field -- pass ARCH=... to build_tissue (see the ARCH cell above)")

# The domain is a VOLUME now, so a flat figure has to choose a plane. field_plane decomposes just
# that plane out of the stored Q-tensor -- decomposing the whole block would be 290 ms of work
# thrown away. Everything below lives on this one plane, support included.
Z0 = TISSUE_VOL[0] // 2
F = field_plane(info["field"], Z0)
# `psi` is now the DISTANCE FROM THE STRUCTURES that the carve ranked by, and `thr` the
# protected collar. They play the role the potential and its quantile did, and read the same way:
# low is near a structure and is kept, high is far and is carved to background.
PB, THR, SUP2 = F["psi"], F["thr"], info["support"][Z0]
LUM2 = F["lumen"]                       # claimed but not tissue -- cells never go here
SNAMES = list(F["term_names"][2:])          # term 0 is the background flow, 1 the outer margin

# MEMBERSHIP is what typing consumes: one soft field per structure, plus whatever is left over as
# background. NOT the director-term responsibilities -- those say which term wins the ORIENTATION,
# which is a different question from which compartment a cell is in.
# The memberships already partition unity (each is discounted by what earlier structures claimed),
# so the background is just the remainder and the stack needs no renormalising.
MB = 1.0 - F["m"].sum(0) if len(F["m"]) else np.ones(SUP2.shape, np.float32)
MRESP = np.concatenate([MB[None], F["m"]], 0)
MNAMES = ["background"] + SNAMES
assert abs(MRESP.sum(0) - 1.0).max() < 1e-4, "memberships must partition unity"

# Line-integral convolution needs FROZEN WHITE noise: tape["dir_a"] is the raw draw, which is
# what keeps the streamlines fine enough to read. (Smoothing it would blur them away.)
LIC = field_lic(F["T"], np.asarray(tape["dir_a"][Z0], float))
ZONE, ZRGB, ZFRAC = zones(MRESP, SUP2)

print("plane z=%d | realised cover %.4f | cells %d | structures: %s"
      % (Z0, info["support"].mean(), len(info["labels_present"]), ", ".join(SNAMES) or "none"))
_rp = info["report"]
print("  lumen %.4f of the frame (claimed, not tissue) | weight solve %d iters, converged=%s"
      % (_rp.get("realised", {}).get("lumen", 0.0), _rp.get("iters", 0), _rp.get("converged")))
# which types actually ended up in each structure -- the profile's effect, measured on the cells
# rather than assumed. Read against FRACTIONS: the marginal moved because the structures decided.
from tissue_direction import sample_field as _sf
_lb = info["labels_present"]
_fc = _sf(info["field"], *[np.array([info["centres_vox"][n][k] for n in _lb]) for k in (0, 1, 2)])
for _si, _nm in enumerate(SNAMES):
    _in = _fc["m"][_si] > 0.9
    if _in.sum():
        print("  inside %-8s %4d cells:" % (_nm, _in.sum()),
              {t: sum(types[n] == t for n, f in zip(_lb, _in) if f) for t in CELLTYPES})
print("  whole frame        %4d cells:" % len(_lb),
      {t: sum(v == t for v in types.values()) for t in CELLTYPES})
print("director tilt: mean |n_z| = %.3f (0 = lying in the section, 1 = through it)"
      % float(F["tilt"][SUP2].mean()))
print("zone areas in this plane: "
      + "  ".join(f"{n} {100*f:.0f}%" for n, f in zip(MNAMES, ZFRAC)))

In [ ]:
NY, NX = TISSUE_VOL[1], TISSUE_VOL[2]
fig, ax = plt.subplots(3, 3, figsize=(18.5, 18.0), 
                     #   facecolor="#0e131a"
                       )


def _dress(a, ttl, sub):
    a.set_xlim(-.5, NX - .5); a.set_ylim(NY - .5, -.5)   # stop overlays padding the axes
    a.set_xticks([]); a.set_yticks([])
    a.set_title(ttl, color="black", fontsize=13, pad=9)
    a.text(.5, -.045, sub, transform=a.transAxes, ha="center", va="top",
           color="black", fontsize=8.6)


def _out(a, c="#e0a43c"):
    """Every structure's REALISED boundary in this plane, and every vessel's lumen wall.

    F["dist"] is d - w, already masked to what each structure actually claimed, so the zero
    contour traces the body where the structure owns it and the INTERFACE where a neighbour met
    it. Contouring the raw distance instead would draw one structure's outline straight through
    another and make two provably disjoint territories look as though they overlap.
    """
    for k in range(len(F["dist"])):
        outline(a, F["dist"][k], float(F["levels"][k]), c, 1.2)      # levels are 0 now
        if F["walls"][k] < 0.0:
            outline(a, F["dist"][k], float(F["walls"][k]), c, 1.7)   # the lumen wall


# --- psi: the potential, and the quantile that becomes the background tissue ---------
a = ax[0, 0]; a.imshow(psi_rgb(PB, THR))
a.contour(PB, levels=[THR], colors="#e0a43c", linewidths=1.4); _out(a)
_dress(a, "Carve score — distance from the structures",
       "the tissue mask is the LOWEST-scoring voxels, so background approaches from far away\n"
       "and stops at the collar  ·  contour = the protected collar  ·  "
       f"realised cover {100*info['support'].mean():.1f}%")

# --- glyphs: the director itself ----------------------------------------------------
a = ax[0, 1]; a.imshow(np.where(SUP2, .14, .055)[..., None] * np.ones(3))
for yy in range(3, NY, 6):
    for xx in range(3, NX, 6):
        if not SUP2[yy, xx]:
            continue
        # length uses S *and* the in-plane projection: a director pointing through the
        # section has almost none, and must draw as a stub rather than a confident segment
        s = F["S"][yy, xx] * float(np.sqrt(max(1.0 - F["tilt"][yy, xx] ** 2, 0.0)))
        ang = F["theta"][yy, xx]
        # NB: never call this L -- the notebook's global L is cfg.L, the spherical-harmonic
        # degree, and a cell-scope assignment here would silently clobber it for build_tissue.
        glen = (1.5 + 4.4 * s) * .5
        a.plot([xx - np.cos(ang) * glen, xx + np.cos(ang) * glen],
               [yy - np.sin(ang) * glen, yy + np.sin(ang) * glen],
               color="#e7e3d8", alpha=float(.20 + .58 * s), lw=1.3)
_out(a)
_dress(a, "Glyphs — directionality (in plane)",
       "length ∝ S × in-plane projection  ·  a stub means the director points THROUGH\n"
       "the section — see the tilt panel  ·  outline = structure boundary, thick = lumen wall")

# --- flow: LIC ----------------------------------------------------------------------
a = ax[0, 2]
_lic = (0.10 + 0.85 * LIC) * np.where(SUP2, 1., .32)
a.imshow(np.clip(_lic[..., None] * (np.array([237, 231, 218]) / 255.), 0, 1)); _out(a)
_dress(a, "Flow — line-integral convolution",
       "frozen noise smeared along the nematic axis  ·  the streamlines cells line up with\n"
       "outside every structure this is the RANDOM background flow, and looks like it")

# --- coherence ----------------------------------------------------------------------
a = ax[1, 0]
a.imshow(np.where(SUP2[..., None], COH_CMAP(F["S"])[..., :3], np.array([10, 14, 20]) / 255.))
_out(a)
_dress(a, "Coherence S", "S = (lam1-lam2)/(1+lam1-lam2) of the Q-tensor  ·  dark 0 → teal → gold 1\n"
       "low S = nothing explains the direction there, so cells barely turn")

# --- how far OUT of the section plane the director points ---------------------------
a = ax[1, 1]
a.imshow(tilt_rgb(F["tilt"], SUP2)); _out(a)
_dress(a, "Director tilt  |n_z|", "0 = lying in the section, 1 = pointing through it\n"
       "the glyph view cannot show this: a steep director draws as a stub")

# --- compartments: SOFT membership --------------------------------------------------
a = ax[1, 2]; a.imshow(np.clip(comp_rgb(MRESP, SUP2), 0, 1)); _out(a, "black")
_dress(a, "Compartments — soft membership", "blended m_s(x):  " + "  ".join(MNAMES)
       + "\nTHIS is what the cell-type profiles are multiplied by")

# --- zones: HARD argmax, the thresholdable regions ----------------------------------
a = ax[2, 0]; a.imshow(ZRGB)
ye, xe = np.nonzero(zone_edges(ZONE, SUP2))
a.scatter(xe, ye, s=.35, c="#0e131a", marker="s", linewidths=0); _out(a, "black")
_dress(a, "Zones — hard argmax", "the regions you can threshold out:  "
       + "  ".join(f"{n} {100*f:.0f}%" for n, f in zip(MNAMES, ZFRAC))
       + "\nMAP-READING ONLY — no argmax ever runs in generator code")

# --- cells --------------------------------------------------------------------------
a = ax[2, 1]
_col = np.where(SUP2[..., None], np.array([24, 31, 41]) / 255., np.array([10, 14, 20]) / 255.)
lab2 = labels[Z0]
tnames = list(CELLTYPES)
_tpal = palette(len(tnames) + 2)[2:]          # skip the flow/margin slots so the hues differ
for n in np.unique(lab2):
    if n:
        _col[lab2 == n] = _tpal[tnames.index(types[int(n)])]
a.imshow(_col)
# Both position and length must come from THIS SLICE, not the cell's 3-D seed. The old
# |seed_z - Z0| < 4 filter drew the glyph at the cell's raw 3-D centre, but a tilted or
# off-centre cell's visible patch at Z0 sits somewhere else entirely -- measured up to ~20 vox
# away for ELONG=6 near the edge of a heavily tilted body -- so the stick floats off the cell it
# belongs to and looks like it points nowhere in particular. Centring on the cell's OWN pixels
# in lab2 fixes that. Length is scaled by sin(POLAR_DEG), the in-plane projection (0 = pointing
# straight through the section) -- the glyphs panel already does this via S*sqrt(1-tilt^2); this
# panel was drawing full length regardless of tilt, which for a steep cell drew a confident-
# looking line where the true in-plane direction is barely defined at all.
_here = [n for n in np.unique(lab2) if n and n in info["geoms"]]
if _here:
    cens, polars, azis = [], [], []
    for n in _here:
        yy, xx = np.nonzero(lab2 == n)
        cens.append((xx.mean(), yy.mean()))
        polars.append(info["geoms"][n].POLAR_DEG.v)
        azis.append(info["geoms"][n].AZIM_DEG.v)
    _cen = np.array(cens)
    _inplane = np.sin(np.deg2rad(polars))
    _azi = np.deg2rad(azis)
    a.quiver(_cen[:, 0], _cen[:, 1], _inplane * np.cos(_azi), _inplane * np.sin(_azi),
             color="#0e131a", headwidth=0, headlength=0, headaxislength=0, pivot="mid",
             scale=30, width=.004)
_out(a, "#f2ead8")
_cnt = {t: sum(types[n] == t for n in info["labels_present"]) for t in tnames}
_dress(a, "Cells — typed and oriented",
       "  ".join(f"{t} {c}" for t, c in _cnt.items())
       + "\nglyph = the cell's own long axis  ·  colours match the profiles above")

# --- zone areas ---------------------------------------------------------------------
a = ax[2, 2]; 
# a.set_facecolor("#141a22")
yv = np.arange(len(MNAMES))[::-1]
a.barh(yv, [100 * f for f in ZFRAC], color=zone_palette(len(MNAMES)), edgecolor="#0e131a")
for i, f in enumerate(ZFRAC):
    a.text(100 * f + 1.5, yv[i], f"{100*f:.0f}%", va="center", color="black", fontsize=11)
a.set_xlim(0, 100); a.set_yticks(yv); a.set_yticklabels(MNAMES, color="black", fontsize=11)
a.set_xticks([]); a.tick_params(colors="black")
for sp in a.spines.values():
    sp.set_color("#2a323d")
a.set_title("Zone areas (this plane)", color="black", fontsize=13, pad=9)
a.text(.5, -.045, "each zone is a region you could threshold out and analyse on its own  ·  "
       "the LEDGER's numbers are for the whole block, printed above",
       transform=a.transAxes, ha="center", va="top", color="black", fontsize=8.6)

plt.tight_layout(rect=[0, .01, 1, .985])
fig.suptitle("Tissue architecture: flow, structures and compartments",
             color="#f2ead8", fontsize=17, y=.999)
plt.show()

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)
tape.drawSensor(shape=(160, 160, 3))

subs = {marker: kryostat(v, optics) for marker, v in t_img.items()}   # keep BOTH vol and z

# We can just use some cell type here as they all use the same channel names
t_psf = np.stack([psf_project(v, z, optics, PANEL.Markers[name].fluorophore)
                  for name, (v, z) in subs.items()], -1)
t_markers = list(subs)
img_adu = detector(t_psf, PANEL, DETECTOR, optics, tape, t_markers)

In [ ]:
sub, sub_z = kryostat(labels, optics)

# plt.imshow(np.sum(sub, axis=0))
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[1])# , cmap = "Purples_r"
plt.axis('off')
plt.show()
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0])# , cmap = "Purples"
plt.axis('off')
plt.show()

In [ ]:
sub, sub_z = kryostat(labels, optics)
sub_n, sub_n_z = kryostat(nuc_labels, optics)


plt.imshow(NormalizeData(img_adu))
# plt.contour(sub[0], colors="red" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green
# " , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99, keep_largest=False)[0], colors="yellow" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

# `lbl`/`lbl_z` (and the nucleus pair) are passed through interact's `fixed(...)` below, so the
# callback binds the tissue label slab captured when THIS cell runs. Without that it would read
# the module-level `sub`, which Section 3's load cell reassigns to a 4D frame batch -- feeding a
# 4D array into mask_collapse then crashes fftconvolve with a dimensionality mismatch.
def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0,
                 show_masks=True,
                 lbl=None, lbl_z=None, lbl_n=None, lbl_n_z=None):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    
    # Checkbox logic
    if show_masks:
        # plt.contour(lbl.any(0), colors="red" , origin="lower", alpha=.55)
        
        plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        # plt.contour(lbl_n.any(0), colors="red" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(lbl_n, lbl_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
         show_masks=Checkbox(value=True, description='Show Masks'),
         # bind the tissue slab now, so a later `sub` reassignment can't reach this callback
         lbl=fixed(sub), lbl_z=fixed(sub_z),
         lbl_n=fixed(sub_n), lbl_n_z=fixed(sub_n_z)
)
print("done")

## 2.1) Introducing Artifacts

In [ ]:
from parameter import Artifacts, ArtifactMap, Fussel, Aggregate, ArtifactMask, Detachment
import artifacts as ART

In [ ]:
AR = Artifacts(
    Map=ArtifactMap(MIN_DIST=12.0, COVER=0.60),
    Fussel=Fussel(N=0, WIDTH_UM=2.5, WOBBLE_UM=7.0),
    Aggregate=Aggregate(N=3, DIAM_UM=1.2, GAIN=120.0),
    Mask=ArtifactMask(DILATE_PX=2.0),
    Detachment=Detachment(ELEVATION_UM=6.0, COVER=0.10, EDGE_BIAS=5., EDGE_WIDTH_UM=7.0),
)

tape.drawArtifacts(shape=TISSUE_VOL, **cfg.ARTIFACTS)

In [ ]:
t_img_art = {k: v.copy() for k, v in t_img.items()}
art = build_artifacts(vols = t_img_art, tape = tape, AR = AR, opt = optics, shape = TISSUE_VOL, panel = PANEL, um_per_vox = UM_PER_VOX, spacing = SPACING, geom=TISSUE_CELL, tissue_support=info["support"], L=L, l_min=L_MIN)


DETACH = ART.detachment_map(tape, AR.Detachment, info["support"], SPACING, UM_PER_VOX, optics.um_per_pz)
SHIFT  = DETACH['shift']
N_EXT  = int(SHIFT.max())
FLAT   = np.zeros_like(SHIFT)


def render_adu(vols, shift=SHIFT, n_ext=N_EXT):
    """kryostat -> displace the cut slab -> psf_project -> detector.
    """
    ss = {m: kryostat(v, optics) for m, v in vols.items()}
    psf = np.stack([ART.lift_project(v, z, shift, n_ext, optics, PANEL.Markers[n].fluorophore)
                    for n, (v, z) in ss.items()], -1)
    return detector(psf, PANEL, DETECTOR, optics, tape, list(ss))

fig, ax = plt.subplots(1, 3, figsize = (16, 8))
adu_art = render_adu(t_img_art)
adu = render_adu(t_img, shift=FLAT, n_ext=0)
sub_art, z_art = kryostat(art["labels"], optics)
sub_art, z_art = ART.lift_slab(sub_art, SHIFT, N_EXT), ART.extend_z(z_art, N_EXT, optics.um_per_pz)
sub_lift, sub_lift_z = ART.lift_slab(sub, SHIFT, N_EXT), ART.extend_z(sub_z, N_EXT, optics.um_per_pz)

ax[0].imshow(NormalizeData(adu_art))
# ax[0].contour(mask_collapse(sub_art, z_art, optics, mask_pct=0.90)[0], colors="violet", origin="lower", alpha=1.)
# ax[0].contour(DETACH["lifted"], colors="orange", origin="lower", alpha=.9)
ax[0].set_title("artifacts (violet = debris, orange = detached)")
ax[1].imshow(NormalizeData(adu))
ax[1].set_title("clean: flat section, no artifacts")
im = ax[2].imshow(DETACH["elevation_um"], cmap="magma")
ax[2].set_title("elevation off the slide (um)")
plt.colorbar(im, ax=ax[2], fraction=0.046)
plt.show()

In [ ]:
img_psf_norm = NormalizeData(np.maximum(adu_art - DETECTOR.OFFSET_ADU.v, 0))

# `lbl`/`lbl_z` (and the nucleus pair) are passed through interact's `fixed(...)` below, so the
# callback binds the tissue label slab captured when THIS cell runs. Without that it would read
# the module-level `sub`, which Section 3's load cell reassigns to a 4D frame batch -- feeding a
# 4D array into mask_collapse then crashes fftconvolve with a dimensionality mismatch.
def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0,
                 show_masks=True, show_art_masks=True,
                 lbl=None, lbl_z=None, sub_art=None, z_art=None):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    
    # Checkbox logic
    if show_masks:
        plt.contour(mask_collapse(lbl, lbl_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        
    if show_art_masks:
        plt.contour(mask_collapse(sub_art, z_art, optics, mask_pct=0.90)[0], colors="violet" , origin="lower", alpha=.55)
        plt.contour(DETACH["lifted"], colors="orange", origin="lower", alpha=.9)
        
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
         show_masks=Checkbox(value=True, description='Show Masks'),
         show_art_masks=Checkbox(value=True, description='Show Artifact Masks'),
         # bind the tissue slab now, so a later `sub` reassignment can't reach this callback
         lbl=fixed(sub), lbl_z=fixed(sub_z),
         sub_art=fixed(sub_art), z_art=fixed(z_art)
)
print("done")

# 3) Inspect a generated dataset

In [ ]:
DATA = Path("../data/example")

_stem = str(DATA.resolve())
if not os.path.exists(_stem + ".npy"):
    raise FileNotFoundError(
        f"no dataset at {_stem}.npy -- generate one first:\n"
        f"    python src/gen_dataset.py make --mode tissue --n 500 --workers 8 --out {_stem}")

meta = np.load(_stem + ".meta.npz", allow_pickle=True)
imgs = np.load(_stem + ".npy", mmap_mode="r")

theta = meta["theta"]
paths = [str(p) for p in meta["paths"]]
n, H, W, C = imgs.shape
# `names`/`dyes` are per channel in image order
names = [str(x) for x in meta["names"]] if "names" in meta.files else \
        [cfg.DAPI_NAME] + [f"ch{k}" for k in range(1, C)]
print(meta["dyes"])
dyes = [str(x) for x in meta["dyes"]] if "dyes" in meta.files else None

print(f"{_stem}")
print(f"  {n} frames, {H}x{W} px, {C} channels, {imgs.dtype} seed={meta['seed']}")
print(f"  {os.path.getsize(_stem + '.npy') / 1e9:.3f} GB on disk, memmapped -- resident cost "
      f"is one frame ({H * W * C * 2 / 1e6:.2f} MB), not the file")
print(f"  theta {theta.shape}: {len(paths)} fitted parameters, normalised to [0, 1]")
print("  channels: " + ", ".join(f"{k}:{nm}" + (f"/{dyes[k]}" if dyes else "")
                                 for k, nm in enumerate(names)))

rng = np.random.default_rng(0)
pick = np.sort(rng.choice(n, size=min(n, 64), replace=False))
sample = np.asarray(imgs[pick], np.float32)

print(f"\nper-channel ADU over {len(pick)} sampled frames "
      f"(black level = DETECTOR OFFSET_ADU = {cfg.DETECTOR['OFFSET_ADU']:.0f})")
print(f"  {'channel':>10}  {'median':>8} {'p99':>8} {'max':>8}   {'at ceiling':>10}")
for k, nm in enumerate(names):
    c = sample[..., k]
    print(f"  {nm:>10}  {np.median(c):8.0f} {np.percentile(c, 99):8.0f} {c.max():8.0f}"
          f"   {100 * (c >= cfg.ADU_MAX).mean():9.3f}%")
    if np.percentile(c, 99) < cfg.DETECTOR["OFFSET_ADU"] + 20:
        print(f"{'':14}^ essentially dark: no cell type expresses {nm}")

In [ ]:
def stretch(frame, p=(1.0, 99.5)):
    
    a = np.asarray(frame, np.float32)
    out = np.zeros(a.shape, np.float32)
    for k in range(a.shape[-1]):
        v0, v1 = np.percentile(a[..., k], p)
        out[..., k] = np.clip((a[..., k] - v0) / max(v1 - v0, 1e-6), 0.0, 1.0)
    return out


def as_rgb(frame):
    """First three channels as RGB, with channel 0 (always DAPI) in BLUE by convention."""
    a = stretch(frame)
    rgb = np.zeros(a.shape[:2] + (3,), np.float32)
    for k in range(min(3, a.shape[-1])):
        rgb[..., 2 - k] = a[..., k]        # ch0 -> blue, ch1 -> green, ch2 -> red
    return rgb


nrow, ncol = 3, 4
show = pick[:nrow * ncol]
fig, axes = plt.subplots(nrow, ncol, figsize=(2.7 * ncol, 2.7 * nrow))
for ax, i in zip(np.ravel(axes), show):
    ax.imshow(as_rgb(imgs[i]))            # one frame off disk, per panel
    ax.set_title(f"#{i}", fontsize=8)
    ax.axis("off")
for ax in np.ravel(axes)[len(show):]:
    ax.axis("off")
_lbl = " / ".join(f"{nm}={c}" for nm, c in zip(names[:3], ("blue", "green", "red")))
fig.suptitle(f"{len(show)} of {n} frames   ({_lbl})", fontsize=10)
plt.tight_layout()
plt.show()

# Every channel of a single frame. This is the one to compare against the notebook's own
# tissue render above -- same forward model, so the textures should be recognisably the same.
i0 = int(show[0])
fig, axes = plt.subplots(1, C, figsize=(3.0 * C, 3.4))
for k, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(stretch(imgs[i0])[..., k], cmap="gray")
    ax.set_title(names[k] + (f"  ({dyes[k]})" if dyes else ""), fontsize=9)
    ax.axis("off")
fig.suptitle(f"frame #{i0}, all {C} channels, 1-99.5 percentile stretch", fontsize=10)
plt.tight_layout()
plt.show()